In [1]:
import os
import numpy as np
from skimage import io
from glob import glob

# === Configuration ===
gt_dir = "/hpc/group/yizhanglab/shared/hDRG_autoseg/processed_data/240819_Ji_N1_H_EScan/pseudo_gt_masks_2048"
pred_base_dir = "../output/infer"

# === Get all ground truth patch files ===
gt_paths = sorted(glob(os.path.join(gt_dir, "patch_col_*_row_*.png")))

ious = []
empty_cases = 0

for gt_path in gt_paths:
    filename = os.path.basename(gt_path)
    pred_dir = os.path.join(pred_base_dir, filename.replace(".png", ""))
    pred_path = os.path.join(pred_dir, "pred_labels.npy")

    if not os.path.exists(pred_path):
        print(f"[SKIP] Missing prediction for {filename}")
        continue

    # Load ground truth mask and convert to binary (1 = object)
    gt_mask = io.imread(gt_path)
    gt_binary = (gt_mask == 255).astype(np.uint8)

    # Load predicted mask and convert to binary (1 = object)
    pred_mask = np.load(pred_path)
    pred_binary = (pred_mask > 0).astype(np.uint8)

    # Check for shape mismatch
    if gt_binary.shape != pred_binary.shape:
        print(f"[SKIP] Shape mismatch for {filename}")
        continue

    # Handle empty case: both masks are completely background
    if np.sum(gt_binary) == 0 and np.sum(pred_binary) == 0:
        iou = 1.0
        empty_cases += 1
    else:
        intersection = np.logical_and(gt_binary, pred_binary).sum()
        union = np.logical_or(gt_binary, pred_binary).sum()
        iou = intersection / union if union > 0 else 0.0

    ious.append(iou)
    print(f"{filename}: IoU = {iou:.4f}")

# === Final report ===
if ious:
    avg_iou = np.mean(ious)
    print("\n=== Summary ===")
    print(f"Evaluated {len(ious)} patches")
    print(f"Average IoU: {avg_iou:.4f}")
    print(f"Empty-background matches (IoU=1): {empty_cases}")
else:
    print("No valid IoUs were computed.")


patch_col_10_row_10.png: IoU = 0.7793
patch_col_10_row_11.png: IoU = 0.6805
patch_col_10_row_12.png: IoU = 0.4631
patch_col_10_row_3.png: IoU = 0.7499
patch_col_10_row_4.png: IoU = 0.6246
patch_col_10_row_5.png: IoU = 0.7048
patch_col_10_row_6.png: IoU = 0.7565
patch_col_10_row_7.png: IoU = 1.0000
patch_col_10_row_8.png: IoU = 0.8293
patch_col_10_row_9.png: IoU = 0.6927
patch_col_11_row_10.png: IoU = 0.7140
patch_col_11_row_11.png: IoU = 0.6877
patch_col_11_row_3.png: IoU = 0.5888
patch_col_11_row_4.png: IoU = 0.5334
patch_col_11_row_5.png: IoU = 0.6077
patch_col_11_row_6.png: IoU = 0.8042
patch_col_11_row_7.png: IoU = 0.7809
patch_col_11_row_8.png: IoU = 0.7574
patch_col_11_row_9.png: IoU = 0.6448
patch_col_12_row_10.png: IoU = 0.7656
patch_col_12_row_11.png: IoU = 0.7230
patch_col_12_row_3.png: IoU = 0.6209
patch_col_12_row_4.png: IoU = 0.5775
patch_col_12_row_5.png: IoU = 0.5643
patch_col_12_row_6.png: IoU = 0.5663
patch_col_12_row_7.png: IoU = 0.3666
patch_col_12_row_8.png: IoU = 0